# Session 6: The Faces Become Data

Click a cell and press **Shift + Enter** to run it. Work from top to bottom.
Cells marked **YOUR TURN** are for you to finish. Run the cells in order.

## 1. Seeds
A seed is a starting number for the computer's random choices. The same seed gives the same choices.

In [ ]:
import random
a = random.Random(42)
b = random.Random(42)
print([round(a.random(), 3) for _ in range(5)])
print([round(b.random(), 3) for _ in range(5)])    # exactly the same
c = random.Random(43)
print([round(c.random(), 3) for _ in range(5)])    # a different seed, different numbers

In [ ]:
import torch
import matplotlib.pyplot as plt
from emoji_faces import make_face, CLASSES

face, mood = make_face(42)
again, same_mood = make_face(42)
print(CLASSES[mood], CLASSES[same_mood])
print("Same seed, same face:", face.tobytes() == again.tobytes())

for seed in [7, 17, 27, 1007]:
    print("seed", seed, "is mood", seed % 10, CLASSES[make_face(seed)[1]])

In [ ]:
# YOUR TURN: which mood will seed 123 be? Work it out first, then check.

## 2. A Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

class EmojiFaces(Dataset):
    def __init__(self, how_many, first_seed, transform=None):
        self.faces = []
        for seed in range(first_seed, first_seed + how_many):
            picture, mood = make_face(seed)
            if transform:
                picture = transform(picture)
            self.faces.append((picture, mood))

    def __len__(self):
        return len(self.faces)

    def __getitem__(self, idx):
        return self.faces[idx]

sample = EmojiFaces(10, first_seed=0)
print(len(sample), "faces in the sample")
picture, mood = sample[3]
print("face number 3 is", CLASSES[mood])

## 3. Getting Pictures Ready

In [ ]:
from torchvision.transforms import v2

get_ready = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

picture, mood = make_face(7)
ready = get_ready(picture)
print(type(picture), "becomes", ready.shape, ready.dtype)
print("smallest:", round(ready.min().item(), 2), " largest:", round(ready.max().item(), 2))

In [ ]:
# YOUR TURN: Normalize works out (value - 0.5) / 0.5. What do 0, 0.25 and 1 become?

## 4. Training Faces, Test Faces and Batches
This draws 15,000 faces: about a minute.

In [ ]:
print("Drawing faces... (about a minute)")
train_faces = EmojiFaces(12000, first_seed=0, transform=get_ready)
test_faces = EmojiFaces(3000, first_seed=1_000_000, transform=get_ready)
train_loader = DataLoader(train_faces, batch_size=64, shuffle=True)
test_loader = DataLoader(test_faces, batch_size=64)
print(len(train_faces), "training faces and", len(test_faces), "test faces")

faces, moods = next(iter(train_loader))
print("one batch of faces:", faces.shape)
print("one batch of moods:", moods.shape)
print("batches in one epoch:", len(train_loader))

In [ ]:
# CHANGE ONE THING: change the batch size, then run.
# PREDICT first: with batches of 100, how many batches will one epoch have?
my_batch_size = 100
loader = DataLoader(train_faces, batch_size=my_batch_size, shuffle=True)
print(len(loader), "batches of", my_batch_size)

In [ ]:
# FINISH THE LINE: replace ______ with True or False, so the training faces come in a new order every epoch.
my_loader = DataLoader(train_faces, batch_size=64, shuffle=______)
print("shuffle is on" if isinstance(my_loader.sampler, torch.utils.data.RandomSampler) else "shuffle is off: try again")

In [ ]:
def show(face_tensor):
    """Draw a face that has been through get_ready."""
    plt.imshow((face_tensor.cpu().permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1))
    plt.axis("off")

plt.figure(figsize=(12, 3.4))
for i in range(16):
    plt.subplot(2, 8, i + 1)
    show(faces[i])
    plt.title(CLASSES[moods[i]], fontsize=9)
plt.show()

In [ ]:
# YOUR TURN: check the training faces are balanced: count how many there are of each mood.
counts = [0] * 10
for _, mood in train_faces:
    ...
print(dict(zip(CLASSES, counts)))